# Temporal Data Analysis — Weekend Gap Prediction

**Dataset:** `dataset_clean_temporal.csv` — 25 tickers, April 2016 – December 2024, weekly Monday rows.  
**Question:** Do cyclic calendar features (`Month_sin/cos`, `WeekOfYear_sin/cos`, `Quarter_sin/cos`, `DaysSinceStart`) improve gap-up prediction beyond the 15-feature technical/fundamental baseline?

---

### Sections
1. Imports & Data Load  
2. EDA — Pre-generated Plots  
3. Temporal Feature Exploration  
4. Seasonal GapUp Patterns  
5. Feature Set Definitions  
6. Walk-forward Model Evaluation  
7. Results & Lift Analysis  
8. Summary  

### Data integrity rules followed throughout
- **COVID exclusion**: `is_extreme_event == 1` rows (Feb–May 2020) dropped from all training and evaluation.
- **No leaky features**: `WeeklyReturn`, `IntraWeekVolatility`, `FridayPosition`, `OpenCloseSpread` never used.
- **Temporal features are calendar-only** — zero price/volume information, zero look-ahead.

---
## 1. Imports & Data Load

### 1.1 Imports

Loading the full stack needed for EDA, statistical tests, and modelling. `scipy.stats.pointbiserialr` gives us the correlation between each continuous temporal feature and the binary `GapUp` label. XGBoost is optional — the notebook runs end-to-end without it, all XGB cells check `_HAS_XGB` before executing.

In [ ]:
import warnings
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
from scipy.stats import pointbiserialr, chi2_contingency
from IPython.display import display

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, accuracy_score, roc_curve

try:
    from xgboost import XGBClassifier
    _HAS_XGB = True
except ImportError:
    _HAS_XGB = False
    warnings.warn('xgboost not installed — XGBoost cells will be skipped.')

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.figsize': (13, 5), 'figure.dpi': 100})
pd.options.display.float_format = '{:.4f}'.format
np.random.seed(42)
%matplotlib inline

print('Imports OK.')
print(f'XGBoost available: {_HAS_XGB}')

### 1.2 Load data and apply primary-set filter

`dataset_clean_temporal.csv` is the standard cleaned dataset with 7 extra calendar columns appended. We drop `is_extreme_event == 1` rows to get the **primary set** — this is the same filter every other model notebook in the project uses, ensuring fair comparison. Timezone is stripped from the `Date` column for cleaner display.

In [ ]:
DATASET_PATH = '../structured_csv_data_files/fetched_data/dataset_clean_temporal.csv'
PLOTS_PATH   = '../structured_csv_data_files/fetched_data/plots/'
TARGET       = 'GapUp'

df_all = pd.read_csv(DATASET_PATH)
df_all['Date'] = pd.to_datetime(df_all['Date'], utc=True).dt.tz_convert(None)

primary = df_all[df_all['is_extreme_event'] == 0].copy()
primary = primary.sort_values(['Ticker', 'Date']).reset_index(drop=True)

print('=== Dataset overview ===')
print(f'  All rows:       {len(df_all):>7,}')
print(f'  Primary rows:   {len(primary):>7,}  (is_extreme_event == 0)')
print(f'  Excluded rows:  {len(df_all) - len(primary):>7,}  (COVID + extreme events)')
print(f'  Tickers:        {primary["Ticker"].nunique():>7}')
print(f'  Date range:     {primary["Date"].min().date()}  to  {primary["Date"].max().date()}')
print(f'  GapUp rate:     {primary[TARGET].mean():.3f}')
print(f'  Total columns:  {len(df_all.columns)}')
print()
print('Columns:')
print(list(df_all.columns))

---
## 2. EDA — Pre-generated Plots

Nine temporal EDA plots were generated separately and saved to `plots/`. We display them here inline. They show the seasonal structure of the dataset and motivate whether calendar features are worth modelling explicitly.

| Figure | What it shows |
|--------|---------------|
| Fig 1 | Annual return distribution — year-over-year spread |
| Fig 2 | Seasonality heatmap — month × year mean gap |
| Fig 3 | Quarterly return profile — Q1–Q4 average |
| Fig 4 | Volatility regime over time |
| Fig 5 | Week-of-year gap rate — which ISO weeks see more GapUps |
| Fig 6 | Ticker × year GapUp heatmap |
| Fig 7 | RSI temporal pattern |
| Fig 8 | Gap extreme events — COVID spike visible |
| Fig 9 | Rolling indicator correlation — feature-target relationship over time |

### 2.1 Render all 9 EDA figures

We scan the `plots/` folder, match each filename to a human-readable title, and display them in order. The key thing to notice across these plots is whether there is **any consistent seasonal structure** — if GapUp rates vary by month or quarter in a repeatable way, the cyclic encodings should help the model.

In [ ]:
plot_meta = [
    ('fig1_annual_return_distribution.png',   'Fig 1 — Annual Return Distribution'),
    ('fig2_seasonality_heatmap.png',           'Fig 2 — Seasonality Heatmap (Month × Year)'),
    ('fig3_quarterly_return_profile.png',      'Fig 3 — Quarterly Return Profile'),
    ('fig4_volatility_regime.png',             'Fig 4 — Volatility Regime Over Time'),
    ('fig5_week_of_year_profile.png',          'Fig 5 — Week-of-Year GapUp Rate Profile'),
    ('fig6_ticker_annual_heatmap.png',         'Fig 6 — Ticker × Year GapUp Heatmap'),
    ('fig7_rsi_temporal.png',                  'Fig 7 — RSI Temporal Pattern'),
    ('fig8_gap_extreme_events.png',            'Fig 8 — Gap Extreme Events'),
    ('fig9_rolling_indicator_correlation.png', 'Fig 9 — Rolling Indicator Correlation'),
]

for fname, title in plot_meta:
    fpath = PLOTS_PATH + fname
    if not os.path.exists(fpath):
        print(f'  [missing] {fname}')
        continue
    img = mpimg.imread(fpath)
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(title, fontsize=13, fontweight='bold', pad=12)
    plt.tight_layout()
    plt.show()

---
## 3. Temporal Feature Exploration

The 7 new features in `dataset_clean_temporal.csv` are all derived from the calendar date — no price or volume data involved:

| Feature | Formula | Range |
|---------|---------|-------|
| `Month_sin` | `sin(2π × month / 12)` | [-1, 1] |
| `Month_cos` | `cos(2π × month / 12)` | [-1, 1] |
| `WeekOfYear_sin` | `sin(2π × week / 52)` | [-1, 1] |
| `WeekOfYear_cos` | `cos(2π × week / 52)` | [-1, 1] |
| `Quarter_sin` | `sin(2π × quarter / 4)` | [-1, 1] |
| `Quarter_cos` | `cos(2π × quarter / 4)` | [-1, 1] |
| `DaysSinceStart` | Days since 2016-04-18 | 0 – ~3,200 |

The sin/cos pairs encode each period **cyclically** — week 52 is numerically close to week 1, which a raw integer would not capture.

### 3.1 Basic stats and missing value check

Before doing anything else, confirm the temporal features are fully populated and look at their distributions. `DaysSinceStart` will have a much larger scale than the cyclic features — this is why the LogReg pipeline needs `StandardScaler`.

In [ ]:
TEMPORAL_FEATURES = [
    'Month_sin', 'Month_cos',
    'WeekOfYear_sin', 'WeekOfYear_cos',
    'Quarter_sin', 'Quarter_cos',
    'DaysSinceStart',
]

print('Missing values:')
print(primary[TEMPORAL_FEATURES].isna().sum().to_string())
print()
print('Descriptive stats:')
display(primary[TEMPORAL_FEATURES].describe().round(3))

### 3.2 Point-biserial correlation with GapUp

Point-biserial correlation is the right measure here — one binary variable (`GapUp`), one continuous variable (each temporal feature). It is mathematically equivalent to Pearson r. A statistically significant correlation (p < 0.05) with a non-trivial |r| tells us the feature carries linear signal. Small |r| values are still useful if they are independent of what the baseline features already capture.

In [ ]:
corr_rows = []
for feat in TEMPORAL_FEATURES:
    r, p = pointbiserialr(primary[TARGET], primary[feat])
    corr_rows.append({
        'Feature':        feat,
        'r':              round(r, 4),
        'p-value':        round(p, 4),
        'Significant':    'yes' if p < 0.05 else 'no',
    })

corr_df = pd.DataFrame(corr_rows).set_index('Feature').sort_values('r', key=abs, ascending=False)
print('Point-biserial correlation — temporal features vs GapUp:')
display(corr_df)

# Visual
fig, ax = plt.subplots(figsize=(9, 4))
colors = ['steelblue' if r > 0 else 'tomato' for r in corr_df['r']]
ax.barh(corr_df.index, corr_df['r'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Point-biserial r')
ax.set_title('Correlation of temporal features with GapUp')
plt.tight_layout()
plt.show()

### 3.3 Distribution of temporal features split by GapUp class

Overlapping density plots show where the two classes (GapUp=0, GapUp=1) differ in the temporal feature space. Wide separation between the orange and blue curves means the feature is discriminative on its own. Even partial separation is useful if it is in a region the baseline features don't cover.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 7))
axes = axes.flatten()

for i, feat in enumerate(TEMPORAL_FEATURES):
    ax = axes[i]
    for label, color, name in [(0, 'steelblue', 'GapUp=0'), (1, 'orange', 'GapUp=1')]:
        vals = primary.loc[primary[TARGET] == label, feat]
        ax.hist(vals, bins=40, alpha=0.5, color=color, label=name, density=True)
    ax.set_title(feat)
    ax.legend(fontsize=7)

axes[-1].axis('off')  # hide the 8th panel (only 7 features)
plt.suptitle('Temporal feature distributions by GapUp class (primary set)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. Seasonal GapUp Patterns

Before modelling, we visualise the raw seasonal signal directly from the data. These charts show whether there are calendar periods where gap-ups are systematically more or less frequent — the model should pick this up if we feed it the right temporal features.

### 4.1 GapUp rate by Month, Quarter, and Year

The red dashed line is the overall mean GapUp rate (~0.516). Bars above it are periods where gap-ups are more frequent than average. We want to see at least some variation — flat bars across all months would mean calendar features add nothing.

In [ ]:
base_rate = primary[TARGET].mean()
fig, axes = plt.subplots(1, 3, figsize=(17, 4))

# By month
mg = primary.groupby('Month')[TARGET].agg(['mean', 'count']).reset_index()
axes[0].bar(mg['Month'], mg['mean'], color='steelblue', alpha=0.85)
axes[0].axhline(base_rate, color='red', linestyle='--', linewidth=1.2, label=f'Mean={base_rate:.3f}')
axes[0].set_xticks(range(1, 13))
axes[0].set_xlabel('Month'); axes[0].set_ylabel('GapUp rate')
axes[0].set_title('By Month'); axes[0].legend(fontsize=8)

# By quarter
qg = primary.groupby('Quarter')[TARGET].mean().reset_index()
axes[1].bar(qg['Quarter'], qg[TARGET], color='darkorange', alpha=0.85)
axes[1].axhline(base_rate, color='red', linestyle='--', linewidth=1.2, label=f'Mean={base_rate:.3f}')
axes[1].set_xlabel('Quarter'); axes[1].set_ylabel('GapUp rate')
axes[1].set_title('By Quarter'); axes[1].legend(fontsize=8)

# By year
yg = primary.groupby('Year')[TARGET].mean().reset_index()
axes[2].bar(yg['Year'].astype(str), yg[TARGET], color='seagreen', alpha=0.85)
axes[2].axhline(base_rate, color='red', linestyle='--', linewidth=1.2, label=f'Mean={base_rate:.3f}')
axes[2].set_xlabel('Year'); axes[2].set_ylabel('GapUp rate')
axes[2].set_title('By Year'); axes[2].legend(fontsize=8)
axes[2].tick_params(axis='x', rotation=45)

plt.suptitle('Seasonal GapUp rates — primary set, COVID excluded', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### 4.2 Week-of-year GapUp rate heatmap

A finer-grained view: GapUp rate for each ISO week of the year, averaged across all years and tickers in the primary set. Strong spikes at specific weeks (e.g. earnings seasons, year-end) would confirm that `WeekOfYear_sin/cos` is worth including.

In [ ]:
woy = primary.groupby('WeekOfYear')[TARGET].agg(['mean', 'count']).reset_index()
woy.columns = ['WeekOfYear', 'GapUp_rate', 'Count']

fig, ax = plt.subplots(figsize=(15, 4))
ax.bar(woy['WeekOfYear'], woy['GapUp_rate'], color='steelblue', alpha=0.7, width=0.8)
ax.axhline(base_rate, color='red', linestyle='--', linewidth=1.2, label=f'Overall mean={base_rate:.3f}')
ax.set_xlabel('ISO Week of Year')
ax.set_ylabel('GapUp rate')
ax.set_title('GapUp rate by week of year (averaged across all tickers and years)')
ax.legend()
plt.tight_layout()
plt.show()

### 4.3 Rolling base-rate trend over time

Has the overall GapUp rate drifted across the 9-year sample? We compute a 52-week × 25-ticker rolling mean (≈1 year of data). If the line drifts significantly, `DaysSinceStart` becomes important as a trend-correction term — and the walk-forward approach is even more critical than a static train/test split.

In [ ]:
trend = primary.sort_values('DaysSinceStart').copy()
trend['rolling_rate'] = trend[TARGET].rolling(window=52 * 25, min_periods=300).mean()

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(trend['Date'], trend['rolling_rate'], color='steelblue', linewidth=1.6, label='Rolling GapUp rate')
ax.axhline(base_rate, color='red', linestyle='--', linewidth=1.2, label=f'Overall mean={base_rate:.3f}')
ax.fill_between(trend['Date'], trend['rolling_rate'], base_rate,
                where=(trend['rolling_rate'] > base_rate), alpha=0.15, color='seagreen', label='Above mean')
ax.fill_between(trend['Date'], trend['rolling_rate'], base_rate,
                where=(trend['rolling_rate'] < base_rate), alpha=0.15, color='tomato', label='Below mean')
ax.set_xlabel('Date'); ax.set_ylabel('Rolling GapUp rate')
ax.set_title('Rolling GapUp rate over time — does the base rate drift? (window ≈ 1 year × all tickers)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

---
## 5. Feature Set Definitions

We test three feature sets head-to-head in the walk-forward evaluation:

| Set | # Features | Contents |
|-----|-----------|----------|
| **Baseline** | 15 | Project-standard technical + fundamental features (post-VIF pruning) |
| **Temporal** | 7 | Calendar-only cyclic encodings + linear trend |
| **Combined** | 22 | Baseline + Temporal |

### 5.1 Define and validate all three feature sets

The baseline 15 match `logistic_regression.ipynb` and `xgboost_model.ipynb` exactly — same post-VIF-pruning selection across momentum, trend, volatility, volume, and fundamental groups. The assertion ensures no NaN values slip through before modelling.

In [ ]:
BASELINE_FEATURES = [
    # Momentum
    'MACD', 'ROC', 'StochPercK',
    # Trend
    'CloseVEma50', 'CloseVSma20', 'ADX',
    # Volatility
    'BollingerBandWidth', 'ATR', 'FiveDStdDev',
    # Volume
    'OBV', 'MFI', 'VolumeRatio',
    # Fundamental
    'NetMargin', 'RoA', 'RevGrowthQoQ',
]

TEMPORAL_FEATURES = [
    'Month_sin', 'Month_cos',
    'WeekOfYear_sin', 'WeekOfYear_cos',
    'Quarter_sin', 'Quarter_cos',
    'DaysSinceStart',
]

COMBINED_FEATURES = BASELINE_FEATURES + TEMPORAL_FEATURES

assert primary[COMBINED_FEATURES].isna().sum().sum() == 0, 'NaN found — check data pipeline'

print(f'Baseline  : {len(BASELINE_FEATURES)} features — {BASELINE_FEATURES}')
print(f'Temporal  : {len(TEMPORAL_FEATURES)} features — {TEMPORAL_FEATURES}')
print(f'Combined  : {len(COMBINED_FEATURES)} features')
print('\nNo NaN values found. Ready to model.')

---
## 6. Walk-forward Model Evaluation

**Fold structure** — expanding window, test year held out each fold:

| Fold | Train | Test | Note |
|------|-------|------|------|
| 1 | 2016–2018 | 2019 | |
| 2 | 2016–2019 | 2020 | COVID fold |
| 3 | 2016–2020 | 2021 | |
| 4 | 2016–2021 | 2022 | |
| 5 | 2016–2022 | 2023 | |
| 6 | 2016–2023 | 2024 | |

Mean AUC is reported over the **5 non-COVID folds** only (2019, 2021–2024).

### 6.1 Walk-forward engine and model factories

A single `walk_forward` function handles all feature sets and classifiers — it accepts a feature list and a model factory, loops over the 6 folds, and returns a tidy DataFrame of per-fold metrics.

- **LogReg** uses `StandardScaler` inside a sklearn `Pipeline`. This is critical because `DaysSinceStart` (scale ~3,000) would otherwise dominate the L2 penalty over the cyclic features (scale [-1, 1]).
- **XGBoost** is scale-invariant — splits use rank order, so no scaler is needed. Settings mirror the project's `xgboost_model.ipynb`.

In [ ]:
FOLDS = [
    {'train': list(range(2016, 2019)), 'test': 2019},
    {'train': list(range(2016, 2020)), 'test': 2020},
    {'train': list(range(2016, 2021)), 'test': 2021},
    {'train': list(range(2016, 2022)), 'test': 2022},
    {'train': list(range(2016, 2023)), 'test': 2023},
    {'train': list(range(2016, 2024)), 'test': 2024},
]

yr_arr = primary['Year'].to_numpy()
y_arr  = primary[TARGET].to_numpy().astype(int)


def walk_forward(features, make_clf, label):
    """Run expanding-window walk-forward evaluation.
    Returns a DataFrame with one row per fold.
    """
    X = primary[features].to_numpy(dtype=float)
    records = []
    for fold in FOLDS:
        tr = np.isin(yr_arr, fold['train'])
        te = yr_arr == fold['test']
        X_tr, X_te = X[tr], X[te]
        y_tr, y_te = y_arr[tr], y_arr[te]
        if len(y_te) == 0 or len(np.unique(y_te)) < 2:
            continue
        clf = make_clf()
        clf.fit(X_tr, y_tr)
        proba = clf.predict_proba(X_te)[:, 1]
        pred  = (proba >= 0.5).astype(int)
        records.append({
            'Label':      label,
            'Test year':  fold['test'],
            'Train rows': int(tr.sum()),
            'Test rows':  int(te.sum()),
            'AUC':        round(roc_auc_score(y_te, proba), 4),
            'Accuracy':   round(accuracy_score(y_te, pred), 4),
            'COVID':      fold['test'] == 2020,
        })
    return pd.DataFrame(records)


def make_logreg():
    return Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(
            penalty='l2', C=1.0, solver='liblinear', max_iter=1000, random_state=42
        )),
    ])


def make_xgb():
    return XGBClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
        eval_metric='auc', tree_method='hist', random_state=42, n_jobs=-1,
    )


print('Walk-forward engine ready.')

### 6.2 Run Logistic Regression across all three feature sets

Three separate walk-forward runs, one per feature set. The pivot table puts all AUC values side-by-side so we can immediately see which set wins each fold and by how much. The two delta columns quantify the lift from adding temporal features.

In [ ]:
print('Running Logistic Regression...')
lr_base = walk_forward(BASELINE_FEATURES, make_logreg, 'LR-Baseline')
lr_temp = walk_forward(TEMPORAL_FEATURES, make_logreg, 'LR-Temporal')
lr_comb = walk_forward(COMBINED_FEATURES, make_logreg, 'LR-Combined')

lr_pivot = pd.DataFrame({
    'Test year':        lr_base['Test year'].values,
    'COVID fold':       lr_base['COVID'].values,
    'Baseline AUC':     lr_base['AUC'].values,
    'Temporal AUC':     lr_temp['AUC'].values,
    'Combined AUC':     lr_comb['AUC'].values,
})
lr_pivot['Δ Comb-Base'] = (lr_pivot['Combined AUC'] - lr_pivot['Baseline AUC']).round(4)
lr_pivot['Δ Temp-Base'] = (lr_pivot['Temporal AUC'] - lr_pivot['Baseline AUC']).round(4)

print('\nLogistic Regression — per-fold AUC:')
display(lr_pivot)

nc = ~lr_base['COVID'].values
print(f'\nMean AUC across non-COVID folds:')
print(f'  Baseline : {lr_base["AUC"].values[nc].mean():.4f}')
print(f'  Temporal : {lr_temp["AUC"].values[nc].mean():.4f}')
print(f'  Combined : {lr_comb["AUC"].values[nc].mean():.4f}')

### 6.3 Run XGBoost across all three feature sets

Same evaluation with XGBoost. Trees can exploit **non-linear interactions** between calendar position and technical indicators (e.g. "high RSI in January" vs "high RSI in October" may have different gap-prediction implications). If the Combined lift is larger here than in LogReg, that interaction signal is real.

In [ ]:
if _HAS_XGB:
    print('Running XGBoost...')
    xgb_base = walk_forward(BASELINE_FEATURES, make_xgb, 'XGB-Baseline')
    xgb_temp = walk_forward(TEMPORAL_FEATURES, make_xgb, 'XGB-Temporal')
    xgb_comb = walk_forward(COMBINED_FEATURES, make_xgb, 'XGB-Combined')

    xgb_pivot = pd.DataFrame({
        'Test year':    xgb_base['Test year'].values,
        'COVID fold':   xgb_base['COVID'].values,
        'Baseline AUC': xgb_base['AUC'].values,
        'Temporal AUC': xgb_temp['AUC'].values,
        'Combined AUC': xgb_comb['AUC'].values,
    })
    xgb_pivot['Δ Comb-Base'] = (xgb_pivot['Combined AUC'] - xgb_pivot['Baseline AUC']).round(4)
    xgb_pivot['Δ Temp-Base'] = (xgb_pivot['Temporal AUC'] - xgb_pivot['Baseline AUC']).round(4)

    print('\nXGBoost — per-fold AUC:')
    display(xgb_pivot)

    nc_x = ~xgb_base['COVID'].values
    print(f'\nMean AUC across non-COVID folds:')
    print(f'  Baseline : {xgb_base["AUC"].values[nc_x].mean():.4f}')
    print(f'  Temporal : {xgb_temp["AUC"].values[nc_x].mean():.4f}')
    print(f'  Combined : {xgb_comb["AUC"].values[nc_x].mean():.4f}')
else:
    print('xgboost not installed — skipping.')
    xgb_base = xgb_temp = xgb_comb = None

---
## 7. Results & Lift Analysis

### 7.1 Per-fold AUC bar charts

Grouped bars: blue = Baseline, orange = Temporal-only, green = Combined. The dotted lines show mean AUC over non-COVID folds. The red shaded column marks the COVID fold (2020) — shown for reference but excluded from all mean calculations. Look for the green bar being consistently above blue to confirm the temporal lift is robust.

In [ ]:
n_plots = 2 if (_HAS_XGB and xgb_base is not None) else 1
fig, axes = plt.subplots(1, n_plots, figsize=(13 * n_plots, 5), sharey=False)
if n_plots == 1:
    axes = [axes]

def _auc_bar_chart(ax, base_df, temp_df, comb_df, title):
    years = base_df['Test year'].tolist()
    x = np.arange(len(years))
    w = 0.26
    ax.bar(x - w, base_df['AUC'], w, label='Baseline', color='steelblue')
    ax.bar(x,     temp_df['AUC'], w, label='Temporal', color='#ff7f0e')
    ax.bar(x + w, comb_df['AUC'], w, label='Combined', color='seagreen')
    ax.axhline(0.5, color='red', linestyle='--', linewidth=1, label='Chance=0.50')
    nc = ~base_df['COVID'].values
    m_base = base_df['AUC'].values[nc].mean()
    m_comb = comb_df['AUC'].values[nc].mean()
    ax.axhline(m_base, color='steelblue', linestyle=':', linewidth=1.2, label=f'Base mean={m_base:.3f}')
    ax.axhline(m_comb, color='seagreen',  linestyle=':', linewidth=1.2, label=f'Comb mean={m_comb:.3f}')
    if 2020 in years:
        cp = years.index(2020)
        ax.axvspan(cp - 0.5, cp + 0.5, alpha=0.08, color='red', label='COVID fold')
    ax.set_xticks(x); ax.set_xticklabels(years)
    ax.set_xlabel('Test year'); ax.set_ylabel('AUC')
    ax.set_title(title); ax.set_ylim(0.38, 0.76)
    ax.legend(loc='upper right', fontsize=8)

_auc_bar_chart(axes[0], lr_base, lr_temp, lr_comb, 'Logistic Regression — AUC by feature set')
if n_plots > 1:
    _auc_bar_chart(axes[1], xgb_base, xgb_temp, xgb_comb, 'XGBoost — AUC by feature set')

plt.tight_layout()
plt.show()

### 7.2 ΔAUC lift chart — Combined minus Baseline per fold

Isolates the **incremental contribution** of temporal features. Each bar shows `AUC(Combined) - AUC(Baseline)` for that test year. Green = temporal features helped; red = they hurt or overfitted. Delta values are printed above each bar for precision. A positive mean across non-COVID folds justifies keeping the temporal features.

In [ ]:
fig, axes = plt.subplots(1, n_plots, figsize=(7 * n_plots, 4), sharey=True)
if n_plots == 1:
    axes = [axes]

def _delta_chart(ax, base_df, comb_df, title):
    deltas = comb_df['AUC'].values - base_df['AUC'].values
    colors = ['seagreen' if d > 0 else 'tomato' for d in deltas]
    years  = base_df['Test year'].astype(str).tolist()
    ax.bar(years, deltas, color=colors, alpha=0.85)
    ax.axhline(0, color='black', linewidth=0.9)
    for i, d in enumerate(deltas):
        ax.text(i, d + (0.0015 if d >= 0 else -0.004), f'{d:+.3f}', ha='center', fontsize=8.5)
    nc = ~base_df['COVID'].values
    mean_d = deltas[nc].mean()
    ax.axhline(mean_d, color='navy', linestyle=':', linewidth=1.2, label=f'Mean (non-COVID)={mean_d:+.4f}')
    ax.set_title(title); ax.set_xlabel('Test year'); ax.set_ylabel('ΔAUC')
    ax.legend(fontsize=8)

_delta_chart(axes[0], lr_base, lr_comb, 'LogReg: ΔAUC (Combined − Baseline)')
if n_plots > 1:
    _delta_chart(axes[1], xgb_base, xgb_comb, 'XGBoost: ΔAUC (Combined − Baseline)')

plt.suptitle('Temporal feature lift over baseline (green = positive)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### 7.3 Final summary table

Mean AUC and mean accuracy over the 5 non-COVID folds for every model × feature-set combination. **This is the single table to cite when comparing results.** The COVID fold (2020) is excluded because its performance is driven by the macro event, not by the feature set.

In [ ]:
summary_rows = []

nc_lr = ~lr_base['COVID'].values
for df in [lr_base, lr_temp, lr_comb]:
    summary_rows.append({
        'Model':             df['Label'].iloc[0],
        'Mean AUC':          round(df['AUC'].values[nc_lr].mean(), 4),
        'Mean Accuracy':     round(df['Accuracy'].values[nc_lr].mean(), 4),
        'Non-COVID folds':   int(nc_lr.sum()),
    })

if _HAS_XGB and xgb_base is not None:
    nc_xgb = ~xgb_base['COVID'].values
    for df in [xgb_base, xgb_temp, xgb_comb]:
        summary_rows.append({
            'Model':           df['Label'].iloc[0],
            'Mean AUC':        round(df['AUC'].values[nc_xgb].mean(), 4),
            'Mean Accuracy':   round(df['Accuracy'].values[nc_xgb].mean(), 4),
            'Non-COVID folds': int(nc_xgb.sum()),
        })

summary = pd.DataFrame(summary_rows)
print('Summary — mean AUC across non-COVID folds (2019, 2021, 2022, 2023, 2024):')
display(summary.sort_values('Mean AUC', ascending=False))

---
## 8. Summary

### How to read the results

| Signal | What it means |
|--------|---------------|
| **Temporal-only AUC > 0.50** | The calendar alone carries gap-prediction signal — earnings cycles, year-end effects, or seasonal repositioning are detectable. |
| **Combined > Baseline** | Temporal features add signal *on top of* the existing technical/fundamental set. The mean ΔAUC over non-COVID folds is the lift to report. |
| **XGB lift > LR lift** | The temporal signal is partly non-linear or interactive (e.g. the effect of high RSI differs by quarter). |
| **Inconsistent fold deltas** | The calendar effect is regime-dependent — real in some years, noise in others. |

### Extension points

- **Earnings-week flag** — a binary `is_earnings_week` derived from quarterly dates would be a sharper temporal signal than cyclic encodings.
- **Ticker-specific seasonality** — some stocks have idiosyncratic earnings calendars; per-ticker month dummies could capture these.
- **Interaction terms** — `Month_sin × RSI`, `Quarter_cos × ADX` — explicitly model conditional seasonality of technical indicators.
- **Compare to KarmaLego/KLS** — `temporal_analysis.ipynb` mines multi-week symbolic patterns (5,245 2-TIRPs, 279K 3-TIRPs); cross-reference that notebook's mean AUC against the Combined mean here to see which temporal representation is stronger.